In [50]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ["OMP_NUM_THREADS"] = '1'
from mpi4py import MPI
import numpy as np
import quimb.tensor as qtn
import quimb as qu
import pickle
from functools import partial
import torch
import json
import autoray as ar
from torch import nn
# ==============================================================================
from vmc_torch.experiment.vmap.vmap_utils import compute_grads, random_initial_config
# from vmc_torch.experiment.vmap.vmap_models import (
#     fPEPS_Model_reuse
# )
from vmc_torch.experiment.vmap.vmap_modules import distributed_minres_solver, run_sampling_phase_reuse
from vmc_torch.hamiltonian_torch import spinful_Fermi_Hubbard_square_lattice_torch
from vmc_torch.experiment.tn_model import init_weights_to_zero
from vmc_torch.experiment.vmap.vmap_torch_utils import robust_svd_err_catcher_wrapper
from vmc_torch.optimizer import DecayScheduler
# ==============================================================================
import warnings
warnings.filterwarnings("ignore")
# ==============================================================================
SVD_JITTER = 1e-16
driver = None
ar.register_function('torch','linalg.svd', lambda x: robust_svd_err_catcher_wrapper(x, jitter=SVD_JITTER, driver=driver))

COMM = MPI.COMM_WORLD
RANK = COMM.Get_rank()
SIZE = COMM.Get_size()
pwd = '/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/data'
torch.set_default_device("cpu")
torch.random.manual_seed(42 + RANK)

def is_vmap_compatible(x):
    """
    Check if a node is compatible with vmap (Tensor or Number).
    """
    return isinstance(x, torch.Tensor)

def is_quimb_place_holder(x):
    return isinstance(x, qu.tensor.interface.Placeholder)

def _get_params_ftn_pytree(ftn):
    ftn_params_raw, _ = qtn.pack(ftn)
    ftn_params = {}

    for key in ftn_params_raw.keys():
        # 1. Convert to raw pytree (contains None, 'Z2', etc.)
        raw_tree = ftn.tensor_map[key].data.to_pytree()
        ftn_params[key] = raw_tree
    return ftn_params

def pack_ftn(ftn):
    # Get raw params and skeleton from Quimb
    ftn_params_raw, skeleton = qtn.pack(ftn)
    ftn_params = {}

    for key in ftn_params_raw.keys():
        # 1. Convert to raw pytree (contains None, 'Z2', etc.)
        raw_tree = ftn.tensor_map[key].data.to_pytree()
        ftn_params[key] = raw_tree
    flat_ftn_params, skeleton_tree = qu.utils.tree_flatten(ftn_params,
                                              get_ref=True,
                                              is_leaf=is_vmap_compatible)
    flat_ftn_params = qu.utils.tree_map(lambda x: torch.as_tensor(x), flat_ftn_params, is_leaf=lambda x: isinstance(x, bool))
    return flat_ftn_params, skeleton


def unpack_ftn(flat_ftn_params, skeleton):
    # Create a shallow copy of the skeleton to modify
    ftn = skeleton.copy()
    ftn_params = _get_params_ftn_pytree(ftn)
    _, pytree = qu.utils.tree_flatten(
        ftn_params,
        get_ref=True,
        is_leaf=lambda x: is_vmap_compatible(x) or is_quimb_place_holder(x),
    )
    ftn_params = qu.utils.tree_unflatten(flat_ftn_params, pytree)
    for key in ftn_params.keys():
        new_data = ftn.tensor_map[key].data.from_pytree(ftn_params[key])
        ftn.tensor_map[key].modify(data=new_data)

    return ftn

def get_params_ftn(ftn):
    flat_ftn_params, _ = pack_ftn(ftn)
    return flat_ftn_params

class fPEPS_Model_reuse(nn.Module):

    def __init__(self, tn, max_bond, dtype=torch.float64, contract_boundary_opts={}, **kwargs):
        import quimb as qu
        import quimb.tensor as qtn
        super().__init__()

        params, skeleton = qtn.pack(tn)
        self.contract_boundary_opts = contract_boundary_opts
        self.dtype = dtype
        self.skeleton = skeleton
        self.Lx = tn.Lx
        self.Ly = tn.Ly
        self.bMPS_x_skeletons = {}
        self.bMPS_y_skeletons = {}
        self.bMPS_params_x_in_dims = None
        self.bMPS_params_y_in_dims = None
        self.chi = max_bond
        self.debug = kwargs.get('debug', False)

        # for torch, further flatten pytree into a single list
        params_flat, params_pytree = qu.utils.tree_flatten(params,
                                                           get_ref=True)
        self.params_pytree = params_pytree

        # register the flat list parameters
        self.params = torch.nn.ParameterList(
            [torch.as_tensor(x, dtype=self.dtype) for x in params_flat])
        
        self.radius = 0

    @torch.no_grad()
    def cache_bMPS_skeleton(self, x):
        params = qu.utils.tree_unflatten(self.params, self.params_pytree)
        tns = qtn.unpack(
            params, self.skeleton
        )  # when unpacking tns, use quimb native unpack function is enough
        amp = tns.isel({
            tns.site_ind(site): x[i]
            for i, site in enumerate(tns.sites)
        })
        env_x = amp.compute_x_environments(max_bond=self.chi, cutoff=0.0, **self.contract_boundary_opts)
        bMPS_params_dict = {}
        for key, tn in env_x.items():
            bMPS_params, skeleton = pack_ftn(tn)
            env_x[key] = skeleton
            bMPS_params_dict[key] = bMPS_params

        self.bMPS_x_skeletons = env_x
        bMPS_params_x_in_dims = qu.utils.tree_map(lambda _: 0,
                                                  bMPS_params_dict)
        self.bMPS_params_x_in_dims = bMPS_params_x_in_dims

        env_y = amp.compute_y_environments(max_bond=self.chi, cutoff=0.0, **self.contract_boundary_opts)
        bMPS_params_dict = {}
        for key, tn in env_y.items():
            bMPS_params, skeleton = pack_ftn(tn)
            env_y[key] = skeleton
            bMPS_params_dict[key] = bMPS_params
        self.bMPS_y_skeletons = env_y
        bMPS_params_y_in_dims = qu.utils.tree_map(lambda _: 0,
                                                  bMPS_params_dict)
        self.bMPS_params_y_in_dims = bMPS_params_y_in_dims
        del env_x
        del env_y
        del bMPS_params_dict


    def cache_bMPS_params_vmap(self, x):
        # return a pytree (dict) of bMPS params for x and y environments
        # For fermions, need to record every tensor's fermionic info too.
        params = qu.utils.tree_unflatten(self.params, self.params_pytree)

        def cache_bMPS_params_single(x_single, params):
            tns = qtn.unpack(
                params, self.skeleton
            )  # when unpacking tns, use quimb native unpack function is enough
            amp = tns.isel({
                tns.site_ind(site): x_single[i]
                for i, site in enumerate(tns.sites)
            })
            env_x = amp.compute_x_environments(max_bond=self.chi, cutoff=0.0, **self.contract_boundary_opts)
            bMPS_params_x_dict = {}
            for key, btn in env_x.items():
                bMPS_params = get_params_ftn(btn)
                bMPS_params_x_dict[key] = bMPS_params
            bMPS_params_y_dict = {}
            env_y = amp.compute_y_environments(max_bond=self.chi, cutoff=0.0, **self.contract_boundary_opts)
            for key, btn in env_y.items():
                bMPS_params = get_params_ftn(btn)
                bMPS_params_y_dict[key] = bMPS_params

            return bMPS_params_x_dict, bMPS_params_y_dict

        return torch.vmap(
            cache_bMPS_params_single,
            in_dims=(0, None),
        )(x, params)

    def cache_bMPS_params_any_direction_vmap(self, x, direction='x'):
        # return a pytree (dict) of bMPS params for x or y environments
        params = qu.utils.tree_unflatten(self.params, self.params_pytree)

        def cache_bMPS_params_x_single(x_single, params):
            tns = qtn.unpack(
                params, self.skeleton
            )  # when unpacking tns, use quimb native unpack function is enough
            amp = tns.isel({
                tns.site_ind(site): x_single[i]
                for i, site in enumerate(tns.sites)
            })
            env_x = amp.compute_x_environments(max_bond=self.chi, cutoff=0.0, **self.contract_boundary_opts)
            amp_val = (env_x[('xmin', self.Lx // 2)]
                       | env_x[('xmax', self.Lx // 2 - 1)]).contract()
            bMPS_params_x_dict = {}
            for key, btn in env_x.items():
                bMPS_params = get_params_ftn(btn)
                bMPS_params_x_dict[key] = bMPS_params
            return bMPS_params_x_dict, amp_val

        def cache_bMPS_params_y_single(x_single, params):
            tns = qtn.unpack(
                params, self.skeleton
            )  # when unpacking tns, use quimb native unpack function is enough
            amp = tns.isel({
                tns.site_ind(site): x_single[i]
                for i, site in enumerate(tns.sites)
            })
            env_y = amp.compute_y_environments(max_bond=self.chi, cutoff=0.0, **self.contract_boundary_opts)
            amp_val = (env_y[('ymin', self.Ly // 2)]
                       | env_y[('ymax', self.Ly // 2 - 1)]).contract()
            bMPS_params_y_dict = {}
            for key, btn in env_y.items():
                bMPS_params = get_params_ftn(btn)
                bMPS_params_y_dict[key] = bMPS_params
            return bMPS_params_y_dict, amp_val

        if direction == 'x':
            return torch.vmap(
                cache_bMPS_params_x_single,
                in_dims=(0, None),
            )(x, params)
        else:
            return torch.vmap(
                cache_bMPS_params_y_single,
                in_dims=(0, None),
            )(x, params)

    def update_bMPS_params_to_row_vmap(self,
                                       x,
                                       row_id,
                                       bMPS_params_x_batched,
                                       from_which='xmin'):
        # update the bMPS params to a specific row_id for all samples in the batch
        bMPS_key = (from_which, row_id)
        params = qu.utils.tree_unflatten(self.params, self.params_pytree)

        def update_bMPS_params_x_single(x_single, params, row_id,
                                        bMPS_params_x, from_which):
            tns = qtn.unpack(params, self.skeleton)
            amp = tns.isel({
                tns.site_ind(site): x_single[i]
                for i, site in enumerate(tns.sites)
            })
            bMPS_to_row = unpack_ftn(bMPS_params_x[bMPS_key],
                                     self.bMPS_x_skeletons[bMPS_key])
            row_tn = amp.select([tns.row_tag(row_id)], which='any')
            # MPO-MPS two row TN
            updated_bMPS = (bMPS_to_row | row_tn)
            # contract to get the updated bMPS, row_id+1 for xmin, row_id-1 for xmax
            if from_which == 'xmin':
                if row_id == 0:
                    updated_bMPS = row_tn
                else:
                    updated_bMPS.contract_boundary_from_xmin_(
                        max_bond=self.chi,
                        cutoff=0.0,
                        xrange=[row_id - 1, row_id],
                        **self.contract_boundary_opts)
                updated_bMPS_params = get_params_ftn(updated_bMPS)
                pytree_params, _ = qu.utils.tree_flatten(updated_bMPS_params,
                                                         get_ref=True)
                _, pytree = qu.utils.tree_flatten(bMPS_params_x[(from_which,
                                                                 row_id + 1)],
                                                  get_ref=True)
                updated_bMPS_params = qu.utils.tree_unflatten(
                    pytree_params, pytree)
                bMPS_params_x[(from_which, row_id +
                               1)] = updated_bMPS_params  # inplace update
            else:
                if row_id == amp.Ly - 1:
                    updated_bMPS = row_tn
                else:
                    updated_bMPS.contract_boundary_from_xmax_(
                        max_bond=self.chi,
                        cutoff=0.0,
                        xrange=[row_id, row_id + 1],
                        **self.contract_boundary_opts)
                updated_bMPS_params = get_params_ftn(updated_bMPS)
                pytree_params, _ = qu.utils.tree_flatten(updated_bMPS_params,
                                                         get_ref=True)
                _, pytree = qu.utils.tree_flatten(bMPS_params_x[(from_which,
                                                                 row_id - 1)],
                                                  get_ref=True)
                updated_bMPS_params = qu.utils.tree_unflatten(
                    pytree_params, pytree)
                bMPS_params_x[(from_which, row_id -
                               1)] = updated_bMPS_params  # inplace update
            return bMPS_params_x

        return torch.vmap(
            update_bMPS_params_x_single,
            in_dims=(0, None, None, self.bMPS_params_x_in_dims, None),
        )(x, params, row_id, bMPS_params_x_batched, from_which)

    def update_bMPS_params_to_col_vmap(self,
                                       x,
                                       col_id,
                                       bMPS_params_y_batched,
                                       from_which='ymin'):
        # update the bMPS params to a specific col_id for all samples in the batch
        bMPS_key = (from_which, col_id)
        params = qu.utils.tree_unflatten(self.params, self.params_pytree)

        def update_bMPS_params_y_single(x_single, params, col_id,
                                        bMPS_params_y, from_which):
            tns = qtn.unpack(params, self.skeleton)
            amp = tns.isel({
                tns.site_ind(site): x_single[i]
                for i, site in enumerate(tns.sites)
            })
            bMPS_to_col = unpack_ftn(bMPS_params_y[bMPS_key],
                                     self.bMPS_y_skeletons[bMPS_key])
            col_tn = amp.select([tns.col_tag(col_id)], which='any')
            # MPO-MPS two col TN
            updated_bMPS = (bMPS_to_col | col_tn)
            # contract to get the updated bMPS, col_id+1 for ymin, col_id-1 for ymax
            if from_which == 'ymin':
                if col_id == 0:
                    updated_bMPS = col_tn
                else:
                    updated_bMPS.contract_boundary_from_ymin_(
                        max_bond=self.chi,
                        cutoff=0.0,
                        yrange=[col_id - 1, col_id],
                        **self.contract_boundary_opts)
                updated_bMPS_params = get_params_ftn(updated_bMPS)
                pytree_params, _ = qu.utils.tree_flatten(updated_bMPS_params,
                                                         get_ref=True)
                _, pytree = qu.utils.tree_flatten(bMPS_params_y[(from_which,
                                                                 col_id + 1)],
                                                  get_ref=True)
                updated_bMPS_params = qu.utils.tree_unflatten(
                    pytree_params, pytree)
                bMPS_params_y[(from_which, col_id +
                               1)] = updated_bMPS_params  # inplace update
            else:
                if col_id == amp.Lx - 1:
                    updated_bMPS = col_tn
                else:
                    updated_bMPS.contract_boundary_from_ymax_(
                        max_bond=self.chi,
                        cutoff=0.0,
                        yrange=[col_id, col_id + 1],
                        **self.contract_boundary_opts)
                updated_bMPS_params = get_params_ftn(updated_bMPS)
                pytree_params, _ = qu.utils.tree_flatten(updated_bMPS_params,
                                                         get_ref=True)
                _, pytree = qu.utils.tree_flatten(bMPS_params_y[(from_which,
                                                                 col_id - 1)],
                                                  get_ref=True)
                updated_bMPS_params = qu.utils.tree_unflatten(
                    pytree_params, pytree)
                bMPS_params_y[(from_which, col_id -
                               1)] = updated_bMPS_params  # inplace update
            return bMPS_params_y

        return torch.vmap(
            update_bMPS_params_y_single,
            in_dims=(0, None, None, self.bMPS_params_y_in_dims, None),
        )(x, params, col_id, bMPS_params_y_batched, from_which)

    def amp_tn(self, x):
        params = qu.utils.tree_unflatten(self.params, self.params_pytree)
        tns = qtn.unpack(params, self.skeleton)
        # might need to specify the right site ordering here
        amp = tns.isel({
            tns.site_ind(site): x[i]
            for i, site in enumerate(tns.sites)
        })
        return amp

    def amplitude(
        self,
        x,
        params,
        bMPS_keys=None,
        bMPS_params_xmin=None,
        bMPS_params_xmax=None,
        bMPS_params_ymin=None,
        bMPS_params_ymax=None,
        selected_rows=None,
        selected_cols=None,
    ):
        tns = qtn.unpack(params, self.skeleton)
        # might need to specify the right site ordering here
        amp = tns.isel({
            tns.site_ind(site): x[i]
            for i, site in enumerate(tns.sites)
        })

        # replace the x-environment with the cached one
        if bMPS_params_xmin is not None and bMPS_params_xmax is not None and bMPS_keys is not None:
            bMPS_min = unpack_ftn(bMPS_params_xmin,
                                  self.bMPS_x_skeletons[bMPS_keys[0]])
            bMPS_max = unpack_ftn(bMPS_params_xmax,
                                  self.bMPS_x_skeletons[bMPS_keys[1]])
            rows = amp.select([tns.row_tag(row) for row in selected_rows],
                              which='any')
            amp_reuse = (bMPS_min | rows | bMPS_max)
            amp_reuse.view_as_(
                qtn.PEPS,
                site_tag_id=tns._site_tag_id,
                x_tag_id=tns._x_tag_id,
                y_tag_id=tns._y_tag_id,
                Lx=tns._Lx,
                Ly=tns._Ly,
                site_ind_id=tns._site_ind_id,
            )
            # if self.chi > 0:
            #     amp_reuse.contract_boundary_from_xmin_(max_bond=self.chi, cutoff=0.0, xrange=[bMPS_keys[0][1], bMPS_keys[1][1]+1])
            return amp_reuse.contract()
        # replace the y-environment with the cached one
        if bMPS_params_ymin is not None and bMPS_params_ymax is not None and bMPS_keys is not None:
            bMPS_min = unpack_ftn(bMPS_params_ymin,
                                  self.bMPS_y_skeletons[bMPS_keys[0]])
            bMPS_max = unpack_ftn(bMPS_params_ymax,
                                  self.bMPS_y_skeletons[bMPS_keys[1]])
            cols = amp.select([tns.col_tag(col) for col in selected_cols],
                              which='any')
            amp_reuse = (bMPS_min | cols | bMPS_max)
            amp_reuse.view_as_(
                qtn.PEPS,
                site_tag_id=tns._site_tag_id,
                x_tag_id=tns._x_tag_id,
                y_tag_id=tns._y_tag_id,
                Lx=tns._Lx,
                Ly=tns._Ly,
                site_ind_id=tns._site_ind_id,
            )
            # if self.chi > 0:
            #     amp_reuse.contract_boundary_from_ymin_(max_bond=self.chi, cutoff=0.0, yrange=[bMPS_keys[0][1], bMPS_keys[1][1]+1])
            return amp_reuse.contract()

        if self.chi > 0:
            amp.contract_boundary_from_ymin_(max_bond=self.chi,
                                             cutoff=0.0,
                                             yrange=[0, amp.Ly // 2 - 1],
                                             **self.contract_boundary_opts)
            amp.contract_boundary_from_ymax_(max_bond=self.chi,
                                             cutoff=0.0,
                                             yrange=[amp.Ly // 2, amp.Ly - 1],
                                             **self.contract_boundary_opts)

        return amp.contract()

    def vamp(
        self,
        x,
        params,
        bMPS_keys=None,
        bMPS_params_xmin=None,
        bMPS_params_xmax=None,
        bMPS_params_ymin=None,
        bMPS_params_ymax=None,
        selected_rows=None,
        selected_cols=None,
    ):
        params = qu.utils.tree_unflatten(params, self.params_pytree)
        if bMPS_params_xmin is not None and bMPS_params_xmax is not None:
            amps = torch.vmap(
                self.amplitude,
                in_dims=(
                    0,
                    None,
                    None,
                    self.bMPS_params_x_in_dims[bMPS_keys[0]],
                    self.bMPS_params_x_in_dims[bMPS_keys[1]],
                    None,
                    None,
                    None,
                    None,
                ),
            )(x, params, bMPS_keys, bMPS_params_xmin, bMPS_params_xmax,
              bMPS_params_ymin, bMPS_params_ymax, selected_rows, selected_cols)
            if self.debug:
                amp_tn_s = [self.amp_tn(x[i]) for i in range(x.shape[0])]
                amps_benchmark = torch.stack([amp_tn.contract() for amp_tn in amp_tn_s], dim=0)
                assert torch.allclose(
                    torch.tensor(amps),
                    torch.tensor(amps_benchmark),
                    rtol=1e-3,
                    atol=1e-3,
                ), f"Amps with bMPS reuse do not match direct contraction! {amps} vs {amps_benchmark}"
            return amps

        if bMPS_params_ymin is not None and bMPS_params_ymax is not None:
            amps = torch.vmap(
                self.amplitude,
                in_dims=(
                    0,
                    None,
                    None,
                    None,
                    None,
                    self.bMPS_params_y_in_dims[bMPS_keys[0]],
                    self.bMPS_params_y_in_dims[bMPS_keys[1]],
                    None,
                    None,
                ),
            )(x, params, bMPS_keys, bMPS_params_xmin, bMPS_params_xmax,
              bMPS_params_ymin, bMPS_params_ymax, selected_rows, selected_cols)
            if self.debug:
                amp_tn_s = [self.amp_tn(x[i]) for i in range(x.shape[0])]
                amps_benchmark = torch.stack([amp_tn.contract() for amp_tn in amp_tn_s], dim=0)
                assert torch.allclose(
                    torch.tensor(amps),
                    torch.tensor(amps_benchmark),
                    rtol=1e-3,
                    atol=1e-3,
                ), f"Amps with bMPS reuse do not match direct contraction! {amps} vs {amps_benchmark}"
            return amps

        return torch.vmap(
            self.amplitude,
            in_dims=(0, None, None, None, None, None, None, None, None),
        )(
            x,
            params,
            bMPS_keys,
            bMPS_params_xmin,
            bMPS_params_xmax,
            bMPS_params_ymin,
            bMPS_params_ymax,
            selected_rows,
            selected_cols,
        )

    def forward(
        self,
        x,
        bMPS_params_x_batched=None,
        bMPS_params_y_batched=None,
        selected_rows=None,
        selected_cols=None,
    ):
        bMPS_params_xmin = None
        bMPS_params_xmax = None
        bMPS_params_ymin = None
        bMPS_params_ymax = None
        bMPS_keys = None

        if selected_rows is not None:
            bMPS_keys = [('xmin', min(selected_rows)),
                         ('xmax', max(selected_rows))]
            bMPS_params_xmin = bMPS_params_x_batched[bMPS_keys[0]]
            bMPS_params_xmax = bMPS_params_x_batched[bMPS_keys[1]]
        if selected_cols is not None:
            bMPS_keys = [('ymin', min(selected_cols)),
                         ('ymax', max(selected_cols))]
            bMPS_params_ymin = bMPS_params_y_batched[bMPS_keys[0]]
            bMPS_params_ymax = bMPS_params_y_batched[bMPS_keys[1]]

        return self.vamp(
            x,
            self.params,
            bMPS_keys=bMPS_keys,
            bMPS_params_xmin=bMPS_params_xmin,
            bMPS_params_xmax=bMPS_params_xmax,
            bMPS_params_ymin=bMPS_params_ymin,
            bMPS_params_ymax=bMPS_params_ymax,
            selected_rows=selected_rows,
            selected_cols=selected_cols,
        )

In [51]:
# ==============================================================================
# 1. Initialization & Configuration
# ==============================================================================
from vmc_torch.experiment.vmap.vmap_modules import sample_next_reuse, evaluate_energy_reuse
Lx, Ly = 4, 4
N_f = Lx * Ly - 2
D, chi = 4, 16
t, U = 1.0, 8.0

# Load PEPS
u1z2 = True
appendix = '_U1SU' if u1z2 else ''
params_path = f'{pwd}/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/Z2/D={D}/'
params = pickle.load(open(params_path + f'peps_su_params{appendix}.pkl', 'rb'))
skeleton = pickle.load(open(params_path + f'peps_skeleton{appendix}.pkl', 'rb'))
peps = qtn.unpack(params, skeleton)
for ts in peps.tensors: 
    ts.modify(data=ts.data.to_flat()*4)
for site in peps.sites: 
    peps[site].data._label = site
    peps[site].data.indices[-1]._linearmap = ((0, 0), (1, 0), (1, 1), (0, 1)) # Important for U1->Z2 fPEPS

# ==============================================================================
# Model Configuration (Define this FIRST)
# ==============================================================================
# 将所有用于初始化的超参数放在这里
# 注意：ftn (peps) 通常太大或是对象，不适合存json，只要记录生成peps的参数(Lx, Ly等)即可
model_config = {
    'max_bond': chi,
    'embed_dim': 16,
    'attn_depth': 1,
    'attn_heads': 4,
    'nn_hidden_dim': 4, #peps.nsites,
    'init_perturbation_scale': 1e-3,
    'nn_eta': 1,
    'dtype_str': 'float64',
    'jitter_svd': 1,
    'uniform_kernel': 0,
}
dtype_map = {'float64': torch.float64, 'float32': torch.float32}
model_dtype = dtype_map[model_config['dtype_str']]
init_kwargs = model_config.copy()
init_kwargs.pop('dtype_str')
# Model
model_reuse = fPEPS_Model_reuse(
    tn=peps,
    max_bond=chi,
    dtype=model_dtype,
    contract_boundary_opts={
        'mode': 'mps',
        # 'equalize_norms': 1.0,
        'canonize': True,
    },
    debug=True,
)
n_params = sum(p.numel() for p in model_reuse.parameters())
if RANK == 0: 
    print(f'Model Params: {n_params}')

# Hamiltonian
H = spinful_Fermi_Hubbard_square_lattice_torch(
    Lx, Ly, t, U, N_f, pbc=False, n_fermions_per_spin=(N_f//2, N_f//2), no_u1_symmetry=False,
)

# VMC Hyperparams
Ns = int(1e3) 
B = 4
B_grad = 4
vmc_steps = 500
init_step = 0
burn_in_steps = 5
learning_rate = 0.1
diag_shift = 1e-4
save_state_every = 10

# Load Checkpoint
file_path = f'{params_path}/{model_reuse._get_name()}/chi={chi}/'
model_reuse.debug_file = file_path
if init_step > 0:
    ckpt_path = file_path + f'checkpoint_step_{init_step}.pt'
    model_reuse.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    if RANK == 0: 
        print(f'Loaded step {init_step}')

# Grad Function
get_grads = partial(compute_grads, vectorize=True, vmap_grad=True, batch_size=B_grad, verbose=False)

# Init State
fxs = torch.stack([random_initial_config(N_f, peps.nsites, seed=42+_) for _ in range(B)]).to(torch.long)
model_reuse.cache_bMPS_skeleton(fxs[0]) if isinstance(model_reuse, fPEPS_Model_reuse) else None
stats = {
    "Np": n_params,
    "sample size": Ns,
    "model_config": model_config,
    "mean": [],
    "error": [],
    "variance": [],
}

# ==============================================================================
# amps check
amp_tns = [model_reuse.amp_tn(test_fx) for test_fx in fxs]
with torch.no_grad():
    B_bMPS_params_x_dict, B_bMPS_params_y_dict = model_reuse.cache_bMPS_params_vmap(fxs)
    current_amps = model_reuse(fxs, bMPS_params_x_batched=B_bMPS_params_x_dict, bMPS_params_y_batched=B_bMPS_params_y_dict, selected_rows=(1,), selected_cols=None)
    amps = torch.tensor([amp_tns[i].contract() for i in range(B)])
# compare relative error from approximate and exact contraction
rel_errs = torch.abs(amps - current_amps) / torch.abs(amps)
print('Relative errors from approximate contraction:', rel_errs)

Model Params: 3200
Relative errors from approximate contraction: tensor([5.9284e-16, 3.5788e-16, 3.8207e-16, 3.2579e-16], dtype=torch.float64)


In [36]:
fxs, current_amps = sample_next_reuse(fxs, model_reuse, H.graph, hopping_rate=0.25, verbose=False)

In [5]:
test_fxs = fxs
amp_tns = [model_reuse.amp_tn(test_fx) for test_fx in test_fxs]
amp = amp_tns[3]

# extract the tensors' data
envs = amp.compute_x_environments(max_bond=chi, cutoff=0.0)
eg_mps_max = envs[('xmax', 0)]
ts_params_pytree_list_max = []
for ts in eg_mps_max.tensors:
    ts_pytree = ts.data.to_pytree()
    ts_params_pytree_list_max.append(ts_pytree)
eg_mps_min = envs[('xmin', 1)]
ts_params_pytree_list_min = []
for ts in eg_mps_min.tensors:
    ts_pytree = ts.data.to_pytree()
    ts_params_pytree_list_min.append(ts_pytree)

# use cached skeleton to create a new bmps (logic reflects reuse of bMPS skeleton in fPEPS_Model_reuse model, explicitly set fermion info here)
new_mps_max = model_reuse.bMPS_x_skeletons[('xmax', 0)].copy()
for i in range(len(new_mps_max.tensors)):
    new_data = new_mps_max.tensors[i].data.from_pytree(ts_params_pytree_list_max[i])
    new_mps_max.tensors[i].modify(data=new_data)
new_mps_min = model_reuse.bMPS_x_skeletons[('xmin', 1)].copy()
for i in range(len(new_mps_min.tensors)):
    new_data = new_mps_min.tensors[i].data.from_pytree(ts_params_pytree_list_min[i])
    new_mps_min.tensors[i].modify(data=new_data)

# naively reuse the bMPS skeleton by copying and setting params (ignored recording fermion info)
naive_mps_max = model_reuse.bMPS_x_skeletons[('xmax', 0)].copy()
mps_max_params = eg_mps_max.get_params()
naive_mps_max.set_params(mps_max_params)
naive_mps_min = model_reuse.bMPS_x_skeletons[('xmin', 1)].copy()
mps_min_params = eg_mps_min.get_params()
naive_mps_min.set_params(mps_min_params)
naive_ts_params_pytree_list_max = []
for ts in naive_mps_max.tensors:
    ts_pytree = ts.data.to_pytree()
    naive_ts_params_pytree_list_max.append(ts_pytree)
naive_ts_params_pytree_list_min = []
for ts in naive_mps_min.tensors:
    ts_pytree = ts.data.to_pytree()
    naive_ts_params_pytree_list_min.append(ts_pytree)

amp.contract(), (new_mps_max|new_mps_min).contract(),  (naive_mps_max|naive_mps_min).contract()
# benchmark value, correct reuse value, naive incorrect reuse value

(tensor(-697.9049, dtype=torch.float64, grad_fn=<MulBackward0>),
 tensor(-697.9049, dtype=torch.float64, grad_fn=<MulBackward0>),
 tensor(23.3170, dtype=torch.float64, grad_fn=<MulBackward0>))

In [150]:
new_mps_max_params = get_params_ftn(new_mps_max)
naive_mps_max_params = get_params_ftn(naive_mps_max)
# compare the params of the two bMPS max tensors
for i in range(len(new_mps_max_params)):
    if isinstance(new_mps_max_params[i], int):
        if not new_mps_max_params[i] == naive_mps_max_params[i]:
            print(f'Int mismatch at {i}')
            print(new_mps_max_params[i], naive_mps_max_params[i])
    if isinstance(new_mps_max_params[i], torch.Tensor):
        if not torch.allclose(new_mps_max_params[i], naive_mps_max_params[i]):
            print(i)
            print(new_mps_max_params[i], naive_mps_max_params[i])

31
tensor([[0, 0],
        [1, 1]]) tensor([[1, 0],
        [0, 1]])
41
tensor([ 1., -1.], dtype=torch.float64) tensor([1., 1.], dtype=torch.float64)
48
tensor(1) tensor(0)


In [151]:
qu.utils.tree_flatten(ts_params_pytree_list_max[0], is_leaf=is_vmap_compatible)

[tensor([[1, 0],
         [0, 1]]),
 tensor([[[ 1.1490e-10, -1.0000e+00],
          [-1.0000e+00, -1.1490e-10]],
 
         [[ 1.0000e+00,  2.2697e-10],
          [-2.2697e-10,  1.0000e+00]]], dtype=torch.float64,
        grad_fn=<MulBackward0>),
 2,
 2,
 True,
 2,
 2,
 False,
 1,
 0,
 6,
 False,
 12,
 False,
 18,
 False,
 1,
 0,
 3,
 False,
 tensor(0),
 2,
 0,
 3,
 False,
 tensor(1),
 3,
 0,
 2,
 False,
 tensor(1)]

In [157]:
ts_params_pytree_list_min[1]

{'sectors': tensor([[0, 0],
         [1, 1]]),
 'blocks': tensor([[[ 2.5673e+00,  1.3333e-14],
          [ 2.7816e-13, -1.9391e-01]],
 
         [[-1.9223e-13,  1.2601e-01],
          [ 4.9941e-01,  4.2048e-14]]], dtype=torch.float64,
        grad_fn=<SelectBackward0>),
 'indices': ({'num_charges': 2,
   'charge_size': 2,
   'dual': True,
   'subinfo': None,
   'linearmap': None},
  {'num_charges': 2,
   'charge_size': 2,
   'dual': False,
   'subinfo': None,
   'linearmap': None}),
 'symmetry': 'Z2',
 'label': (0, 1),
 'phases': tensor([1, 1]),
 'dummy_modes': ({'label': 3, 'dual': False, 'parity': np.int64(1)},
  {'label': ('squeeze', (0, 1), 2), 'dual': False, 'parity': tensor(1)})}

In [158]:
naive_ts_params_pytree_list_min[1]

{'sectors': tensor([[0, 0],
         [1, 1]]),
 'blocks': tensor([[[ 2.5673e+00,  1.3333e-14],
          [ 2.7816e-13, -1.9391e-01]],
 
         [[-1.9223e-13,  1.2601e-01],
          [ 4.9941e-01,  4.2048e-14]]], dtype=torch.float64,
        grad_fn=<SelectBackward0>),
 'indices': ({'num_charges': 2,
   'charge_size': 2,
   'dual': True,
   'subinfo': None,
   'linearmap': None},
  {'num_charges': 2,
   'charge_size': 2,
   'dual': False,
   'subinfo': None,
   'linearmap': None}),
 'symmetry': 'Z2',
 'label': (0, 1),
 'phases': tensor([1, 1]),
 'dummy_modes': ({'label': 3, 'dual': False, 'parity': np.int64(1)},
  {'label': ('squeeze', (0, 1), 2), 'dual': False, 'parity': tensor(1)})}

In [185]:
def is_vmap_compatible(x):
    """
    Check if a node is compatible with vmap (Tensor or Number).
    """
    return isinstance(x, torch.Tensor)

def is_quimb_place_holder(x):
    return isinstance(x, qu.tensor.interface.Placeholder)

def _get_params_ftn_pytree(ftn):
    ftn_params_raw, _ = qtn.pack(ftn)
    ftn_params = {}

    for key in ftn_params_raw.keys():
        # 1. Convert to raw pytree (contains None, 'Z2', etc.)
        raw_tree = ftn.tensor_map[key].data.to_pytree()
        ftn_params[key] = raw_tree
    return ftn_params

def pack_ftn(ftn):
    # Get raw params and skeleton from Quimb
    ftn_params_raw, skeleton = qtn.pack(ftn)
    ftn_params = {}

    for key in ftn_params_raw.keys():
        # 1. Convert to raw pytree (contains None, 'Z2', etc.)
        raw_tree = ftn.tensor_map[key].data.to_pytree()
        ftn_params[key] = raw_tree
    flat_ftn_params, skeleton_tree = qu.utils.tree_flatten(ftn_params,
                                              get_ref=True,
                                              is_leaf=is_vmap_compatible)
    flat_ftn_params = qu.utils.tree_map(lambda x: torch.as_tensor(x), flat_ftn_params, is_leaf=lambda x: isinstance(x, bool))
    return flat_ftn_params, skeleton


def unpack_ftn(flat_ftn_params, skeleton):
    # Create a shallow copy of the skeleton to modify
    ftn = skeleton.copy()
    ftn_params = _get_params_ftn_pytree(ftn)
    _, pytree = qu.utils.tree_flatten(
        ftn_params,
        get_ref=True,
        is_leaf=lambda x: is_vmap_compatible(x) or is_quimb_place_holder(x),
    )
    ftn_params = qu.utils.tree_unflatten(flat_ftn_params, pytree)
    for key in ftn_params.keys():
        new_data = ftn.tensor_map[key].data.from_pytree(ftn_params[key])
        ftn.tensor_map[key].modify(data=new_data)

    return ftn

def get_params_ftn(ftn):
    flat_ftn_params, _ = pack_ftn(ftn)
    return flat_ftn_params



amp = amp_tns[0]
flat_ftn_params, skeleton = pack_ftn(amp)
amp1 = amp_tns[1]
flat_ftn_params1, skeleton1 = pack_ftn(amp1)
new_amp_reconstructed = unpack_ftn(flat_ftn_params, skeleton1)
new_amp1_reconstructed = unpack_ftn(flat_ftn_params1, skeleton)
amp.contract(), new_amp_reconstructed.contract(), amp1.contract(), new_amp1_reconstructed.contract()

(tensor(-5.1134, dtype=torch.float64, grad_fn=<MulBackward0>),
 tensor(-5.1134, dtype=torch.float64, grad_fn=<MulBackward0>),
 tensor(11.5093, dtype=torch.float64, grad_fn=<MulBackward0>),
 tensor(11.5093, dtype=torch.float64, grad_fn=<MulBackward0>))

In [177]:
for i in range(len(flat_ftn_params)):
    # if isinstance(flat_ftn_params[i], torch.Tensor):
    #     if not torch.allclose(flat_ftn_params[i], flat_ftn_params1[i]):
    #         print(f'Tensor type diff at {i}: {flat_ftn_params[i]} vs {flat_ftn_params1[i]}')
    if isinstance(flat_ftn_params[i], int):
        if flat_ftn_params[i] != flat_ftn_params1[i]:
            print(f'Int type diff at {i}: {flat_ftn_params[i]} vs {flat_ftn_params1[i]}')
    

In [171]:
B_bMPS_params_x_dict[('xmax', 0)]
# ts_params_pytree_list

[tensor([[[0, 0],
          [1, 1]],
 
         [[0, 0],
          [1, 1]],
 
         [[0, 0],
          [1, 1]],
 
         [[0, 0],
          [1, 1]]]),
 tensor([[[[ 2.1946e+00, -1.7168e-16],
           [ 5.1457e-19, -7.2527e-02]],
 
          [[-1.7594e+00,  8.7334e-15],
           [ 1.4855e-14,  8.7197e-01]]],
 
 
         [[[ 9.1837e-01, -1.2537e-16],
           [-6.6808e-18, -1.7263e-01]],
 
          [[-1.2121e+00, -9.5981e-15],
           [ 1.0411e-14, -8.6327e-01]]],
 
 
         [[[ 2.1946e+00, -1.7168e-16],
           [ 5.1457e-19, -7.2527e-02]],
 
          [[-1.7594e+00,  8.7334e-15],
           [ 1.4855e-14,  8.7197e-01]]],
 
 
         [[[ 2.1946e+00, -1.7168e-16],
           [ 5.1457e-19, -7.2527e-02]],
 
          [[-1.7594e+00,  8.7334e-15],
           [ 1.4855e-14,  8.7197e-01]]]], dtype=torch.float64),
 tensor([2, 2, 2, 2]),
 tensor([2, 2, 2, 2]),
 tensor([True, True, True, True]),
 tensor([2, 2, 2, 2]),
 tensor([2, 2, 2, 2]),
 tensor([False, False, False, False]),

# symmray version 01/22: 85ab16c

In [31]:
model_reuse.cache_bMPS_skeleton(fxs[0]) if isinstance(model_reuse, fPEPS_Model_reuse) else None
bMPS_xmax_0_skeleton = model_reuse.bMPS_x_skeletons[('xmax', 1)]
bMPS_xmin_0_skeleton = model_reuse.bMPS_x_skeletons[('xmin', 2)]
for ts in bMPS_xmax_0_skeleton.tensors:
    print(ts.data.dummy_modes)
print('---')
model_reuse.cache_bMPS_skeleton(fxs[1]) if isinstance(model_reuse, fPEPS_Model_reuse) else None
bMPS_xmax_1_skeleton = model_reuse.bMPS_x_skeletons[('xmax', 1)]
bMPS_xmin_1_skeleton = model_reuse.bMPS_x_skeletons[('xmin', 2)]
for ts in bMPS_xmax_1_skeleton.tensors:
    print(ts.data.dummy_modes)

(24-, 36-, ('squeeze', (2, 0), 3)-, ('squeeze', (3, 0), 2)-)
(39-, ('squeeze', (2, 1), 4)-, ('squeeze', (3, 1), 3)-)
(30-, 42-, ('squeeze', (2, 2), 4)-, ('squeeze', (3, 2), 3)-)
(33-, 45-, ('squeeze', (2, 3), 3)-, ('squeeze', (3, 3), 2)-)
---
(24-, 36-, ('squeeze', (2, 0), 3)-, ('squeeze', (3, 0), 2)-)
(39-, ('squeeze', (2, 1), 4)-, ('squeeze', (3, 1), 3)-)
(30-, 42-, ('squeeze', (2, 2), 4)-, ('squeeze', (3, 2), 3)-)
(33-, 45-, ('squeeze', (2, 3), 3)-, ('squeeze', (3, 3), 2)-)


In [32]:
# model_reuse.bMPS_params_x_in_dims
single_bMPS_params_x_dict = B_bMPS_params_x_dict.copy()
single_bMPS_params_x_dict = torch.utils._pytree.tree_map(lambda x: x[0], single_bMPS_params_x_dict)

bMPS_xmax_0_skeleton.set_params(single_bMPS_params_x_dict[('xmax', 1)])
bMPS_xmin_0_skeleton.set_params(single_bMPS_params_x_dict[('xmin', 2)])
bMPS_xmax_1_skeleton.set_params(single_bMPS_params_x_dict[('xmax', 1)])
bMPS_xmin_1_skeleton.set_params(single_bMPS_params_x_dict[('xmin', 2)])

(bMPS_xmax_0_skeleton|bMPS_xmin_0_skeleton).contract(), (bMPS_xmax_1_skeleton|bMPS_xmin_1_skeleton).contract()

(tensor(-28.6076, dtype=torch.float64), tensor(1.9002, dtype=torch.float64))

In [33]:
(bMPS_xmax_0_skeleton|bMPS_xmin_1_skeleton).contract(), (bMPS_xmax_1_skeleton|bMPS_xmin_0_skeleton).contract()

(tensor(0., dtype=torch.float64), tensor(0., dtype=torch.float64))

In [34]:
torch.allclose(bMPS_xmax_0_skeleton.arrays[2].blocks, bMPS_xmax_1_skeleton.arrays[2].blocks)

True

In [36]:
(bMPS_xmax_0_skeleton|bMPS_xmin_0_skeleton).contract(), (bMPS_xmax_1_skeleton|bMPS_xmin_0_skeleton).contract()

(tensor(-28.6076, dtype=torch.float64), tensor(0., dtype=torch.float64))

In [37]:
bMPS_xmax_0_skeleton.arrays, bMPS_xmax_1_skeleton.arrays

((Z2FermionicArrayFlat(shape~(4, 4):[-+], charge=0, num_blocks=2, dummy_modes=(24-, 36-, ('squeeze', (2, 0), 3)-, ('squeeze', (3, 0), 2)-)),
  Z2FermionicArrayFlat(shape~(4, 4, 16):[--+], charge=1, num_blocks=4, dummy_modes=(39-, ('squeeze', (2, 1), 4)-, ('squeeze', (3, 1), 3)-)),
  Z2FermionicArrayFlat(shape~(4, 16, 4):[--+], charge=0, num_blocks=4, dummy_modes=(30-, 42-, ('squeeze', (2, 2), 4)-, ('squeeze', (3, 2), 3)-)),
  Z2FermionicArrayFlat(shape~(4, 4):[--], charge=1, num_blocks=2, dummy_modes=(33-, 45-, ('squeeze', (2, 3), 3)-, ('squeeze', (3, 3), 2)-))),
 (Z2FermionicArrayFlat(shape~(4, 4):[-+], charge=0, num_blocks=2, dummy_modes=(24-, 36-, ('squeeze', (2, 0), 3)-, ('squeeze', (3, 0), 2)-)),
  Z2FermionicArrayFlat(shape~(4, 4, 16):[--+], charge=1, num_blocks=4, dummy_modes=(39-, ('squeeze', (2, 1), 4)-, ('squeeze', (3, 1), 3)-)),
  Z2FermionicArrayFlat(shape~(4, 16, 4):[--+], charge=0, num_blocks=4, dummy_modes=(30-, 42-, ('squeeze', (2, 2), 4)-, ('squeeze', (3, 2), 3)-)),
  

In [65]:
for ts in bMPS_xmax_0_skeleton.tensors:
    print(ts.data.to_pytree())

{'sectors': tensor([[0, 0],
        [1, 1]]), 'blocks': tensor([[[ 1.0000e+00, -1.1792e-10],
         [-1.1792e-10, -1.0000e+00]],

        [[ 2.8961e-06, -1.0000e+00],
         [ 1.0000e+00,  2.8961e-06]]], dtype=torch.float64), 'indices': ({'num_charges': 2, 'charge_size': 2, 'dual': True, 'subinfo': None, 'linearmap': None}, {'num_charges': 2, 'charge_size': 2, 'dual': False, 'subinfo': None, 'linearmap': None}), 'symmetry': 'Z2', 'label': None, 'phases': tensor([1., 1.], dtype=torch.float64), 'dummy_modes': ({'label': 24, 'dual': False, 'parity': np.int64(1)}, {'label': 36, 'dual': False, 'parity': np.int64(1)}, {'label': ('squeeze', (2, 0), 3), 'dual': False, 'parity': tensor(1)}, {'label': ('squeeze', (3, 0), 2), 'dual': False, 'parity': tensor(1)})}
{'sectors': tensor([[0, 0, 1],
        [1, 1, 1],
        [0, 1, 0],
        [1, 0, 0]]), 'blocks': tensor([[[[-8.9601e-01, -1.9612e-09, -1.8652e-05, -4.3867e-01,  3.4423e-04,
            2.1488e-04,  6.8831e-02,  2.1888e-04],
      

In [66]:
for ts in bMPS_xmax_1_skeleton.tensors:
    print(ts.data.to_pytree())

{'sectors': tensor([[0, 0],
        [1, 1]]), 'blocks': tensor([[[ 1.0000e+00, -1.1792e-10],
         [-1.1792e-10, -1.0000e+00]],

        [[ 2.8961e-06, -1.0000e+00],
         [ 1.0000e+00,  2.8961e-06]]], dtype=torch.float64), 'indices': ({'num_charges': 2, 'charge_size': 2, 'dual': True, 'subinfo': None, 'linearmap': None}, {'num_charges': 2, 'charge_size': 2, 'dual': False, 'subinfo': None, 'linearmap': None}), 'symmetry': 'Z2', 'label': None, 'phases': tensor([1., 1.], dtype=torch.float64), 'dummy_modes': ({'label': 24, 'dual': False, 'parity': np.int64(1)}, {'label': 36, 'dual': False, 'parity': np.int64(1)}, {'label': ('squeeze', (2, 0), 3), 'dual': False, 'parity': tensor(1)}, {'label': ('squeeze', (3, 0), 2), 'dual': False, 'parity': tensor(1)})}
{'sectors': tensor([[0, 0, 1],
        [1, 1, 1],
        [0, 1, 0],
        [1, 0, 0]]), 'blocks': tensor([[[[-8.9601e-01, -1.9612e-09, -1.8652e-05, -4.3867e-01,  3.4423e-04,
            2.1488e-04,  6.8831e-02,  2.1888e-04],
      

In [528]:
changed_row_id = 1
# randomly permute the row 0 of all test_fxs
# torch.random.manual_seed(42)
# perm = torch.randperm(Ly)
# test_fxs[:, changed_row_id*Ly:(changed_row_id+1)*Ly] = test_fxs[:, changed_row_id*Ly:(changed_row_id+1)*Ly][:, perm]
# print(test_fxs)

with torch.no_grad():
    new_B_bMPS_params_x_dict = model_reuse.update_bMPS_params_to_row_vmap(test_fxs, row_id=1, bMPS_params_x_batched=B_bMPS_params_x_dict, from_which='xmin')

# get the pytree structure of new_B_bMPS_params_x_dict[('xmin', 2)]
with torch.no_grad():
    pytree = torch.utils._pytree.tree_flatten(new_B_bMPS_params_x_dict[('xmin', 2)])[1]
    pytree_old = torch.utils._pytree.tree_flatten(B_bMPS_params_x_dict[('xmin', 2)])[1]
    current_amps = model_reuse(test_fxs, bMPS_params_x_batched=new_B_bMPS_params_x_dict, bMPS_params_y_batched=B_bMPS_params_y_dict, selected_rows=(2,), selected_cols=None)
# current_amps
# print("Amplitudes with reuse after updating row 1:", current_amps)

In [ ]:
import mpi4py.MPI as MPI
import time
COMM = MPI.COMM_WORLD
RANK = COMM.Get_rank()

# cache skeleton for reuse
model_reuse.cache_bMPS_skeleton(test_fxs[0]) 
@torch.inference_mode()
def evaluate_energy_reuse(fxs, v_model, H, current_amps, verbose=False, benchmark_model=None):
    t0 = time.time()
    # Label each connected config with which sample it comes from to enable reuse
    B = fxs.shape[0]
    # cache bMPS params for reuse
    B_bMPS_params_x_dict, B_bMPS_params_y_dict = v_model.cache_bMPS_params_vmap(fxs)

    def detect_changed_row_col_pair(fx1, fx2):
        # currently only support nearest neighbor on square lattice
        Ly = v_model.skeleton._Ly
        changed_pos = torch.nonzero(fx1 - fx2)
        changed_pos_2d = []
        assert changed_pos.shape[0] <= 2, "Expect at most 2 on-site config changes"
        for pos in changed_pos:
            x, y = pos.item() // Ly, pos.item() % Ly
            changed_pos_2d.append( (x, y) )
        if len(changed_pos_2d) == 2:
            delta_row = abs(changed_pos_2d[0][0] - changed_pos_2d[1][0])
            delta_col = abs(changed_pos_2d[0][1] - changed_pos_2d[1][1])
            if delta_row <= delta_col:
                x1 = min(changed_pos_2d, key=lambda t: t[0])[0]
                row = True
                col = False
                return row, col, list(x for x in range(x1, x1+delta_row+1))
            else:
                y1 = min(changed_pos_2d, key=lambda t: t[1])[1]
                row = False
                col = True
                return row, col, list(y for y in range(y1, y1+delta_col+1))
        else:
            row = col = False
            return row, col, None
             
    # --------------------------------------------------------------------------
    # 2. Batched Calculation with Reuse
    # --------------------------------------------------------------------------
    # get connected configurations, coefficients and indices
    conn_eta_num = []
    conn_etas = []
    conn_eta_coeffs = []
    conn_eta_indices = []

    fx_ind = 0
    for fx in fxs:
        eta, coeffs = H.get_conn(fx)
        for i_eta in range(len(eta)):
            r, c, pos = detect_changed_row_col_pair(fx, eta[i_eta])
            conn_eta_indices.append( (fx_ind, i_eta, r, c, pos) )

        conn_eta_num.append(len(eta))
        conn_etas.append(torch.tensor(eta))
        conn_eta_coeffs.append(torch.tensor(coeffs))
        fx_ind += 1

    conn_etas = torch.cat(conn_etas, dim=0)
    conn_eta_coeffs = torch.cat(conn_eta_coeffs, dim=0)

    # group connected configs by changed row/col first
    # within row/col group, further group by position

    # select the indices where r==True and c==False
    # TODO: modify this part, form the pytree with batched leaves to input to reusbale PEPS amplitude calculation [x]
    tasks_map = {} # Key: (mode, indices_tuple), Value: lists of (global_idx, parent_idx)
    
    # 记录哪些是“对角项”（x' == x），这些不需要重算，直接复用当前振幅
    diagonal_mask = torch.zeros(conn_etas.shape[0], dtype=torch.bool, device=fxs.device)
    diagonal_parent_indices = []
    
    # 遍历所有连接构型，进行归类
    # conn_eta_indices[k] = (parent_fx_ind, i_eta, r_bool, c_bool, pos_list)
    for k, (parent_idx, _, is_row, is_col, indices) in enumerate(conn_eta_indices):
        if indices is None:
            # Config 没有变化 (Diagonal term, e.g. density-density interaction)
            diagonal_mask[k] = True
            diagonal_parent_indices.append(parent_idx)
            continue
            
        mode = 'row' if is_row else 'col'
        # 将 list 转为 tuple 以便作为 dict key
        indices_tuple = tuple(sorted(indices)) 
        group_key = (mode, indices_tuple)
        
        if group_key not in tasks_map:
            tasks_map[group_key] = {'global_idxs': [], 'parent_idxs': []}
        
        tasks_map[group_key]['global_idxs'].append(k)
        tasks_map[group_key]['parent_idxs'].append(parent_idx)

    # --------------------------------------------------------------------------
    # 2. Batched Calculation with Reuse
    # --------------------------------------------------------------------------
    # 预分配结果容器
    total_conns = conn_etas.shape[0]
    conn_amps = torch.zeros(total_conns, dtype=current_amps.dtype, device=fxs.device)
    
    # A. 处理对角项 (Direct Copy)
    if len(diagonal_parent_indices) > 0:
        # 找出所有对角项在 conn_amps 中的位置
        diag_locs = torch.nonzero(diagonal_mask).squeeze()
        # 找出对应的 parent index
        parents = torch.tensor(diagonal_parent_indices, device=fxs.device)
        # 直接赋值: <x|psi>
        conn_amps[diag_locs] = current_amps[parents]

    # B. 处理非对角项 (Grouped Vmap Contraction)
    # Fetch the corresponding Batched Environment Dictionary in the pytree
    def slice_env_dict(env_dict, idxs):
        """
        env_dict: {key: PyTree_of_Tensors}
        idxs: indices to slice (tensor or list)
        
        我们需要对 env_dict 中的每一个 value (PyTree) 进行操作，
        利用 qu.utils.tree_map 进入 PyTree 内部，对每个叶子 Tensor 进行切片。
        """
        return {
            k: qu.utils.tree_map(lambda x: x[idxs], v) 
            for k, v in env_dict.items()
        }

    for (mode, indices), data in tasks_map.items():
        # 获取索引
        global_idxs = data['global_idxs']  # 在 conn_etas 中的位置
        parent_idxs = data['parent_idxs']  # 来源于哪个 fxs[i]
        
        # 构造当前 Group 的 Batch
        # 1. 目标构型 x'
        # 注意：conn_etas 是一个大 tensor，直接切片即可
        target_configs = conn_etas[global_idxs] # (Batch_Group, N_sites)
        
        # 2. 对应的父辈索引 (用于取环境)
        subset_parents = torch.tensor(parent_idxs, device=fxs.device)
        
        # 3. 切片环境参数
        subset_env_x = slice_env_dict(B_bMPS_params_x_dict, subset_parents)
        subset_env_y = slice_env_dict(B_bMPS_params_y_dict, subset_parents)
        
        # 4. 调用模型 (Reuse Forward)
        # 这里利用了 v_model.forward 的接口
        if mode == 'row':
            amps_group = v_model(
                target_configs,
                bMPS_params_x_batched=subset_env_x,
                bMPS_params_y_batched=subset_env_y,
                selected_rows=indices,
                selected_cols=None
            )
        else: # col
            amps_group = v_model(
                target_configs,
                bMPS_params_x_batched=subset_env_x,
                bMPS_params_y_batched=subset_env_y,
                selected_rows=None,
                selected_cols=indices
            )
            
        # 5. 填回结果 Tensor
        locs = torch.tensor(global_idxs, device=fxs.device)
        conn_amps[locs] = amps_group

    # --------------------------------------------------------------------------
    # 3. Compute Local Energy
    # --------------------------------------------------------------------------
    # E_loc(x) = \sum_{x'} H_{x,x'} * (psi(x') / psi(x))
    
    # 此时 conn_amps 已经填满，且顺序与 conn_etas 一致
    # conn_eta_num[b] 告诉我们每个样本 b 有多少个连接构型
    
    local_energies = []
    offset = 0
    
    # 为了避免显式 Python 循环 (虽然 B=1024 时循环也不慢)，可以使用 segment_coo 或者简单的 loop
    # 简单 loop 实现：
    for b in range(B):
        n_conn = conn_eta_num[b]
        
        # 取出当前样本相关的所有连接构型的振幅
        # shape: (n_conn,)
        amps_slice = conn_amps[offset : offset + n_conn]
        coeffs_slice = conn_eta_coeffs[offset : offset + n_conn]
        
        # Ratio: psi(x') / psi(x)
        # current_amps[b] 是标量
        ratio = amps_slice / current_amps[b]
        
        # H_loc = \sum H_{xx'} * Ratio
        e_loc = torch.sum(coeffs_slice * ratio)
        
        local_energies.append(e_loc)
        offset += n_conn
        
    local_energies = torch.stack(local_energies)
    
    # Global Mean Energy
    energy_mean = torch.mean(local_energies)

    # if verbose and torch.distributed.is_initialized() and torch.distributed.get_rank() == 0:
    #     print(f"Reuse Stats: Processed {len(tasks_map)} groups + diagonal terms.")

    if benchmark_model is not None and verbose:
        t1 = time.time()
        print(f'Energy (Reuse)    : {energy_mean.item()}, time: {t1 - t0:.4f} s')
        # below is logics without reuse implemented yet
        ################################################################################
        current_amps_benchmark = benchmark_model(fxs)
        t0 = time.time()
        # calculate amplitudes for connected configs, in the future consider TN reuse to speed up calculation, TN reuse is controlled by a param that is not batched over (control flow?)
        conn_amps_benchmark = torch.cat([benchmark_model(conn_etas[i:i+B]) for i in range(0, conn_etas.shape[0], B)])

        # Local energy \sum_{s'} H_{s,s'} <s'|psi>/<s|psi>

        local_energies = []
        offset = 0
        for b in range(B):
            n_conn = conn_eta_num[b]
            amps_ratio = conn_amps_benchmark[offset:offset+n_conn] / current_amps_benchmark[b]
            energy_b = torch.sum(conn_eta_coeffs[offset:offset+n_conn] * amps_ratio)
            local_energies.append(energy_b)
            offset += n_conn
        local_energies = torch.stack(local_energies, dim=0)
        # Energy: (1/N) * \sum_s <s|H|psi>/<s|psi> = (1/N) * \sum_s \sum_{s'} H_{s,s'} <s'|psi>/<s|psi>
        energy = torch.mean(local_energies)
        t1 = time.time()
        print(f'Energy (Benchmark): {energy.item()}, time: {t1 - t0:.4f} s')
        ################################################################################

    return energy_mean, local_energies
    

energy_mean, local_energies = evaluate_energy_reuse(
    test_fxs, model_reuse, H, current_amps, verbose=True, benchmark_model=model
)
print(f'mean energy std: {local_energies.std().item()/np.sqrt(reuse_B)}')

/tmp/ipykernel_1386/881000176.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  changed_pos = torch.nonzero(fx1 - fx2)


Energy (Reuse)    : -19.618847917040775, time: 2.1739 s
Energy (Benchmark): -19.618848144724087, time: 6.0998 s
mean energy std: 0.2823158582856436


In [541]:
import mpi4py.MPI as MPI
import time
import random
from vmc_torch.experiment.vmap.vmap_utils import propose_exchange_or_hopping, sample_next
COMM = MPI.COMM_WORLD
RANK = COMM.Get_rank()

@torch.inference_mode()
def sample_next_reuse(fxs, v_model, graph, hopping_rate=0.25, verbose=False, seed=None, benchmark_model=None):
    B = fxs.shape[0]
    # cache bMPS params along x direction first for reuse
    B_bMPS_params_x_dict, current_amps = v_model.cache_bMPS_params_any_direction_vmap(fxs, direction='x')
    for row, edges in graph.row_edges.items():
        for edge in edges:
            i, j = edge
            proposed_fxs, new_flags = [], []
            fx_id = 0
            for fx in fxs:
                proposed_fx, new = propose_exchange_or_hopping(
                    i,
                    j,
                    fx,
                    hopping_rate=hopping_rate,
                    seed=seed + fx_id if seed is not None else None,
                )
                proposed_fxs.append(proposed_fx)
                new_flags.append(new)
                fx_id += 1
            proposed_fxs = torch.stack(proposed_fxs, dim=0)
            if not any(new_flags):
                if verbose:
                    print(f"No changes proposed in this edge. (x, {row}, edge: {edge})")
                continue
            # compute amplitudes for all proposed configs (because must align batchsize with reused bMPS batchsize)
            new_proposed_fxs = proposed_fxs
            # reuse of bMPS from xmin & xmax w.r.t. row to compute amplitudes
            new_proposed_amps = v_model(
                new_proposed_fxs,
                bMPS_params_x_batched=B_bMPS_params_x_dict,
                bMPS_params_y_batched=None,
                selected_rows=(row,),
                selected_cols=None,
            )
            proposed_amps = new_proposed_amps
            if benchmark_model is not None and verbose:
                benchmark_amps = benchmark_model(new_proposed_fxs)
                print(f"Benchmark vs Reuse amplitudes check (x, {row}):")
                print(torch.allclose(benchmark_amps, new_proposed_amps, atol=1e-5))
            # Metropolis-Hastings acceptance
            ratio = proposed_amps**2 / current_amps**2
            accept_probs = torch.minimum(ratio, torch.ones_like(ratio))
            for k in range(B):
                if random.random() < accept_probs[k].item():
                    fxs[k] = proposed_fxs[k]
                    current_amps[k] = proposed_amps[k]
        
        # update bMPS params to next row for reuse
        if row == v_model.Lx - 1:
            # reach the bottom row, no further bMPS to be updated
            break
        B_bMPS_params_x_dict = v_model.update_bMPS_params_to_row_vmap(
            fxs,
            row_id = row,
            bMPS_params_x_batched=B_bMPS_params_x_dict,
            from_which='xmin',
        )

    B_bMPS_params_y_dict, current_amps = v_model.cache_bMPS_params_any_direction_vmap(fxs, direction='y')
    for col, edges in graph.col_edges.items():
        for edge in edges:
            i, j = edge
            proposed_fxs, new_flags = [], []
            fx_id = 0
            for fx in fxs:
                proposed_fx, new = propose_exchange_or_hopping(
                    i,
                    j,
                    fx,
                    hopping_rate=hopping_rate,
                    seed=seed + fx_id if seed is not None else None,
                )
                proposed_fxs.append(proposed_fx)
                new_flags.append(new)
                fx_id += 1
            proposed_fxs = torch.stack(proposed_fxs, dim=0)
            if not any(new_flags):
                if verbose:
                    print(f"No changes proposed in this edge. (y, {col}, edge: {edge})")
                continue
            # compute amplitudes for all proposed configs (because must align batchsize with reused bMPS batchsize)
            new_proposed_fxs = proposed_fxs
            # reuse of bMPS from ymin & ymax w.r.t. col to compute amplitudes
            new_proposed_amps = v_model(
                new_proposed_fxs,
                bMPS_params_x_batched=None,
                bMPS_params_y_batched=B_bMPS_params_y_dict,
                selected_rows=None,
                selected_cols=(col,),
            )
            if benchmark_model is not None and verbose:
                benchmark_amps = benchmark_model(new_proposed_fxs)
                print(f"Benchmark vs Reuse amplitudes check (y, {col}):")
                print(torch.allclose(benchmark_amps, new_proposed_amps, atol=1e-5))
            proposed_amps = new_proposed_amps
            # Metropolis-Hastings acceptance
            ratio = proposed_amps**2 / current_amps**2
            accept_probs = torch.minimum(ratio, torch.ones_like(ratio))
            for k in range(B):
                if random.random() < accept_probs[k].item():
                    fxs[k] = proposed_fxs[k]
                    current_amps[k] = proposed_amps[k]
        
        # update bMPS params to next col for reuse
        if col == v_model.Ly - 1:
            # reach the rightmost col, no further bMPS to be updated
            break
        B_bMPS_params_y_dict = v_model.update_bMPS_params_to_col_vmap(
            fxs,
            col_id = col,
            bMPS_params_y_batched=B_bMPS_params_y_dict,
            from_which='ymin',
        )

    return fxs, current_amps

t0 = time.time()
test_fxs, current_amps = sample_next_reuse(
    test_fxs,
    model_reuse,
    H.graph,
    hopping_rate=0.0,
    verbose=False,
    seed=42,
    benchmark_model=model
)
t1 = time.time()
print(f"Sampling with reuse done. Time: {t1 - t0:.4f} s")

t0 = time.time()
test_fxs, current_amps = sample_next(
    test_fxs,
    model,
    H.graph,
    hopping_rate=0.0,
    verbose=False,
    seed=42,
)
t1 = time.time()
print(f"Sampling benchmark done. Time: {t1 - t0:.4f} s")

Sampling with reuse done. Time: 3.1913 s
Sampling benchmark done. Time: 6.3457 s


In [10]:
_EXTRA_PROPS = ('_site_tag_id', '_x_tag_id', '_y_tag_id', '_Lx', '_Ly', '_site_ind_id')
# get the extra properties of the tensor network
qtn_extra_props = {prop: getattr(peps, prop) for prop in _EXTRA_PROPS}
qtn_extra_props

{'_site_tag_id': 'I{},{}',
 '_x_tag_id': 'X{}',
 '_y_tag_id': 'Y{}',
 '_Lx': 6,
 '_Ly': 6,
 '_site_ind_id': 'k{},{}'}